# 16단계: 배제된 데이터(44.62%)가 편향됐는지 확인

각 셀을 순서대로 실행하세요. 무거운 셀은 시간이 걸릴 수 있습니다(진행 로그가 찍힙니다).


In [1]:
import sys, os
sys.path.append(os.path.abspath("."))   # 00_config.py, utils.py 가 있는 폴더 (이 노트북과 같은 폴더에 두세요)
from importlib import import_module

import pandas as pd
from scipy.stats import chi2_contingency

config = import_module("00_config")
utils = import_module("utils")

FLAG_THRESHOLD = 15.0  # 이 값(%p) 이상 편차 나면 '편향 의심'으로 표시


## 1. shop_period_raw.parquet 로딩 및 배제여부 계산

In [2]:
t0 = utils.start_timer()
utils.log("shop_period_raw.parquet 로딩 시작...")
df = pd.read_parquet(
    config.out_path("shop_period_raw.parquet"),
    columns=["시군구명", "period", "상권업종대분류명", "PNU19", "층정보_신뢰가능여부"],
)
utils.log(f"로딩 완료 ({len(df):,}행)", t0)

df["배제여부"] = df["층정보_신뢰가능여부"] != "검증완료"
overall_rate = df["배제여부"].mean()
print(f"전체 평균 배제율: {overall_rate*100:.2f}%")


[시작] shop_period_raw.parquet 로딩 시작...
  [18.0초 경과] 로딩 완료 (19,079,683행)
전체 평균 배제율: 44.62%


## 2. 축별 배제율 확인 함수

이 함수를 아래 3, 4, 5번 셀에서 계속 재사용합니다.


In [3]:
def bias_by_axis(df, axis_col, overall_rate, label):
    print(f"\n{'='*10} [{label}] {'='*10}")
    stat = (
        df.groupby(axis_col)
        .agg(전체건수=("배제여부", "size"), 배제건수=("배제여부", "sum"))
        .assign(배제율=lambda d: (d["배제건수"] / d["전체건수"] * 100).round(2))
    )
    stat["편차"] = (stat["배제율"] - overall_rate * 100).round(2)
    stat = stat.sort_values("편차", ascending=False)
    print(stat.to_string())

    try:
        crosstab = pd.crosstab(df[axis_col], df["배제여부"])
        chi2, p, dof, _ = chi2_contingency(crosstab)
        print(f"\n카이제곱 통계량: {chi2:,.1f} | p-value: {p:.2e} | 자유도: {dof}")
        if p < 0.05:
            print("-> 통계적으로 유의미한 차이 (표본이 크면 흔한 결과이니 '편차'(%p) 크기를 더 중시)")
    except Exception as e:
        print(f"카이제곱 검정 실패(무시 가능): {e}")

    return stat


## 3. 축 1 — 지역(시군구명)별 배제율

In [4]:
region_stat = bias_by_axis(df, "시군구명", overall_rate, "지역별 배제율")
region_stat.to_csv(config.out_path("bias_by_region.csv"), encoding="utf-8-sig")



========== [지역별 배제율] ==========
         전체건수    배제건수    배제율     편차
시군구명                               
구로구    657756  353023  53.67   9.05
금천구    596931  313729  52.56   7.94
영등포구   990780  519226  52.41   7.79
중구     923207  464527  50.32   5.70
종로구    766210  385292  50.29   5.67
노원구    569583  274844  48.25   3.63
송파구   1180198  555758  47.09   2.47
서초구   1359781  632406  46.51   1.89
성동구    585181  266330  45.51   0.89
용산구    560690  251735  44.90   0.28
중랑구    539907  240958  44.63   0.01
강동구    677408  301831  44.56  -0.06
동대문구   620752  276294  44.51  -0.11
동작구    507518  223475  44.03  -0.59
강서구    893621  391080  43.76  -0.86
성북구    565979  244878  43.27  -1.35
관악구    708477  297487  41.99  -2.63
양천구    570254  237056  41.57  -3.05
서대문구   504145  208143  41.29  -3.33
강남구   2169191  893387  41.19  -3.43
도봉구    377721  153574  40.66  -3.96
광진구    652401  264217  40.50  -4.12
강북구    468180  185890  39.70  -4.92
마포구   1029385  369440  35.89  -8.73
은평구    604427  209088  34.59 -1

## 4. 축 2 — 건물 주용도별 배제율

In [5]:
utils.log("building_agg.parquet 로딩 및 주용도 결합 시작...")
building_agg = pd.read_parquet(config.out_path("building_agg.parquet"), columns=["주용도_대표"])
df["주용도_대표"] = df["PNU19"].map(building_agg["주용도_대표"])
utils.log("결합 완료", t0)

use_stat = bias_by_axis(df, "주용도_대표", overall_rate, "건물 주용도별 배제율")
use_stat.to_csv(config.out_path("bias_by_building_use.csv"), encoding="utf-8-sig")


[시작] building_agg.parquet 로딩 및 주용도 결합 시작...
  [58.3초 경과] 결합 완료

========== [건물 주용도별 배제율] ==========
               전체건수     배제건수     배제율     편차
주용도_대표                                     
다가구주택            15       15  100.00  55.38
묘지관련시설          219      196   89.50  44.88
위험물저장및처리시설    26899    22991   85.47  40.85
장례시설            269      224   83.27  38.65
운수시설          39758    32461   81.65  37.03
분뇨.쓰레기처리시설      671      534   79.58  34.96
수련시설           1733     1335   77.03  32.41
관광휴게시설         1334     1021   76.54  31.92
야영장시설           161      107   66.46  21.84
판매시설         792262   520909   65.75  21.13
동물및식물관련시설       346      223   64.45  19.83
자동차관련시설      122508    77473   63.24  18.62
창고시설          23642    14396   60.89  16.27
숙박시설         133772    80190   59.95  15.33
교육연구및복지시설      1989     1191   59.88  15.26
문화및집회시설       52174    30949   59.32  14.70
판매및영업시설       38040    22321   58.68  14.06
공장           643297   373294   58.03  13.41
종교시설          15327 

## 5. 축 3 — 연도별 배제율

In [6]:
df["연도"] = df["period"] // 100
year_stat = bias_by_axis(df, "연도", overall_rate, "연도별 배제율")
year_stat.to_csv(config.out_path("bias_by_year.csv"), encoding="utf-8-sig")



========== [연도별 배제율] ==========
         전체건수    배제건수    배제율    편차
연도                                
2015   349035  178280  51.08  6.46
2016  1565716  778539  49.72  5.10
2017  1301247  626578  48.15  3.53
2018  1809288  847487  46.84  2.22
2019  1873470  857015  45.74  1.12
2020  1949584  873511  44.80  0.18
2021  2042383  899260  44.03 -0.59
2022  2122936  921460  43.40 -1.22
2025  2147869  906842  42.22 -2.40
2023  1936994  807051  41.67 -2.95
2024  1981161  817645  41.27 -3.35

카이제곱 통계량: 55,962.1 | p-value: 0.00e+00 | 자유도: 10
-> 통계적으로 유의미한 차이 (표본이 크면 흔한 결과이니 '편차'(%p) 크기를 더 중시)


## 6. 배제 전/후 구성비 변화

"배제 전(원본 전체)"과 "배제 후(검증완료만)"를 비교해서, 업종/지역 구성비가 얼마나 달라졌는지 확인합니다.


In [7]:
utils.log("배제 전/후 업종 구성비 비교 시작...")
before_ind = df["상권업종대분류명"].value_counts(normalize=True) * 100
after_ind = df.loc[~df["배제여부"], "상권업종대분류명"].value_counts(normalize=True) * 100
comp_ind = pd.DataFrame({"배제전_비중": before_ind, "배제후_비중": after_ind})
comp_ind["변화(%p)"] = (comp_ind["배제후_비중"] - comp_ind["배제전_비중"]).round(2)
comp_ind = comp_ind.round(2).sort_values("변화(%p)")
print("[배제 전/후 업종 구성비 변화]")
print(comp_ind.to_string())
comp_ind.to_csv(config.out_path("composition_shift_industry.csv"), encoding="utf-8-sig")


[시작] 배제 전/후 업종 구성비 비교 시작...
[배제 전/후 업종 구성비 변화]
          배제전_비중  배제후_비중  변화(%p)
상권업종대분류명                        
소매         22.62   18.45   -4.17
과학·기술      15.32   14.26   -1.07
시설관리·임대     4.22    3.62   -0.59
숙박          1.49    0.97   -0.51
부동산         5.16    5.00   -0.16
수리·개인       9.73    9.60   -0.13
보건의료        3.05    3.52    0.47
교육          6.59    7.59    1.01
예술·스포츠      4.09    5.15    1.06
음식         27.73   31.83    4.10


In [8]:
before_reg = df["시군구명"].value_counts(normalize=True) * 100
after_reg = df.loc[~df["배제여부"], "시군구명"].value_counts(normalize=True) * 100
comp_reg = pd.DataFrame({"배제전_비중": before_reg, "배제후_비중": after_reg})
comp_reg["변화(%p)"] = (comp_reg["배제후_비중"] - comp_reg["배제전_비중"]).round(2)
comp_reg = comp_reg.round(2).sort_values("변화(%p)")
print("[배제 전/후 지역 구성비 변화]")
print(comp_reg.to_string())
comp_reg.to_csv(config.out_path("composition_shift_region.csv"), encoding="utf-8-sig")
utils.log("비교 완료", t0)


[배제 전/후 지역 구성비 변화]
      배제전_비중  배제후_비중  변화(%p)
시군구명                        
영등포구    5.19    4.46   -0.73
구로구     3.45    2.88   -0.56
중구      4.84    4.34   -0.50
금천구     3.13    2.68   -0.45
종로구     4.02    3.61   -0.41
송파구     6.19    5.91   -0.28
서초구     7.13    6.88   -0.24
노원구     2.99    2.79   -0.20
성동구     3.07    3.02   -0.05
용산구     2.94    2.92   -0.01
중랑구     2.83    2.83   -0.00
강동구     3.55    3.55    0.00
동대문구    3.25    3.26    0.01
동작구     2.66    2.69    0.03
강서구     4.68    4.76    0.07
성북구     2.97    3.04    0.07
도봉구     1.98    2.12    0.14
서대문구    2.64    2.80    0.16
양천구     2.99    3.15    0.16
관악구     3.71    3.89    0.18
강북구     2.45    2.67    0.22
광진구     3.42    3.67    0.25
은평구     3.17    3.74    0.57
강남구    11.37   12.07    0.71
마포구     5.40    6.25    0.85
  [86.0초 경과] 비교 완료


## 7. 최종 종합 판정 (편차 {FLAG_THRESHOLD}%p 이상인 카테고리 요약)

In [9]:
print(f"[최종 종합 판정: 편차 {FLAG_THRESHOLD}%p 이상 카테고리]")
any_flag = False
for name, stat in [("지역", region_stat), ("건물주용도", use_stat), ("연도", year_stat)]:
    flagged = stat[stat["편차"].abs() >= FLAG_THRESHOLD]
    if len(flagged) > 0:
        any_flag = True
        print(f"\n[{name}] 편향 의심 카테고리:")
        print(flagged[["전체건수", "배제율", "편차"]].to_string())

if not any_flag:
    print(f"\n어느 축에서도 {FLAG_THRESHOLD}%p 이상 벌어지는 카테고리가 없습니다.")
    print("-> 배제가 특정 범주에 심하게 쏠리지 않고, 비교적 고르게 퍼져있다고 볼 수 있습니다.")
else:
    print(f"\n위 카테고리들은 배제율이 전체 평균과 {FLAG_THRESHOLD}%p 이상 차이 나므로,")
    print("해당 범주에 대한 분석 결과는 표본 편향 가능성을 염두에 두고 해석해야 합니다.")

utils.log("전체 완료", t0)


[최종 종합 판정: 편차 15.0%p 이상 카테고리]

[건물주용도] 편향 의심 카테고리:
              전체건수     배제율     편차
주용도_대표                           
다가구주택           15  100.00  55.38
묘지관련시설         219   89.50  44.88
위험물저장및처리시설   26899   85.47  40.85
장례시설           269   83.27  38.65
운수시설         39758   81.65  37.03
분뇨.쓰레기처리시설     671   79.58  34.96
수련시설          1733   77.03  32.41
관광휴게시설        1334   76.54  31.92
야영장시설          161   66.46  21.84
판매시설        792262   65.75  21.13
동물및식물관련시설      346   64.45  19.83
자동차관련시설     122508   63.24  18.62
창고시설         23642   60.89  16.27
숙박시설        133772   59.95  15.33
교육연구및복지시설     1989   59.88  15.26
교정및군사시설        662   25.83 -18.79
공공용시설           32    0.00 -44.62
교정시설            40    0.00 -44.62
국방,군사시설          6    0.00 -44.62

위 카테고리들은 배제율이 전체 평균과 15.0%p 이상 차이 나므로,
해당 범주에 대한 분석 결과는 표본 편향 가능성을 염두에 두고 해석해야 합니다.
  [86.1초 경과] 전체 완료
